# Experiment 2: NLP Applications and Mini-Project Problem Statement
**Mini-project:** Resume Analyzer Using Natural Language Processing (Group 28)

**Objective:** Explore NLP applications, review related work and formulate the mini-project problem statement.

Open in Colab: [Exp2_Problem_Statement.ipynb](https://colab.research.google.com/github/GnsShubham06/resume-analyzer-nlp/blob/main/notebooks/experiments/Exp2_Problem_Statement.ipynb) &nbsp;|&nbsp; Run with **Runtime → Run all**.
Data: 100 resumes (20 per domain); the screenshots in the report use five fictional (synthetic) resumes.

In [ ]:
# ---- Setup: runs in Google Colab and locally --------------------------------
# In Colab this cell downloads the project repository (code + anonymised
# dataset) from GitHub and installs the packages Colab does not ship with.
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
REPO = "https://github.com/GnsShubham06/resume-analyzer-nlp"
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/resume-analyzer-nlp"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/resume-analyzer-nlp"], check=True)
    os.chdir("/content/resume-analyzer-nlp")
    extra = []
    if extra:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
else:  # local run from notebooks/experiments/
    for _ in range(3):
        if os.path.isfile("requirements.txt"):
            break
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for pkg in []:
    nltk.download(pkg, quiet=True)

%config InlineBackend.figure_format = "retina"
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.read_csv("data/resume_dataset.csv")
df["extracted_text"] = df["extracted_text"].fillna("")
print("Project folder:", os.path.basename(os.getcwd()), "| resumes:", len(df))

## Problem statement
Given a resume (PDF, DOCX or text), the system should **(A)** classify it into one of five domains –
Engineering, HR, Finance, Healthcare or Agriculture – and **(B)** extract skills, education, job titles and
organisations, after removing personal details.

| | |
|---|---|
| Input | an English resume |
| Output | predicted domain + confidence, structured JSON profile |
| NLP tasks | text categorization, information extraction (NER) |
| Evaluation | accuracy and macro-F1 on held-out resumes, compared with a TF-IDF baseline |

In [ ]:
# Five fictional (synthetic) resumes used for the screenshots - one per domain
SAMPLE_IDS = {"Engineering": "ENG_06", "HR": "HR_06", "Finance": "FIN_07",
              "Healthcare": "HLT_04", "Agriculture": "AGR_02"}
FOOTER = "Synthetic resume - fictional person"
def resume_text(rid):
    """Raw text of a fictional resume, without the 'synthetic resume' footer line."""
    t = df.set_index("resume_id").loc[rid, "extracted_text"]
    return "\n".join(l for l in t.split("\n") if not l.startswith(FOOTER)).strip()

import re
# Our own keyword list per domain (illustration of text categorization, not a learned model)
KEYWORDS = {
    "Engineering": ["design", "cad", "solidworks", "ansys", "finite element", "manufacturing", "prototype",
                    "tolerance", "welding", "plc", "electrical", "mechanical", "drawings", "engineer", "gd&t"],
    "HR": ["recruitment", "onboarding", "payroll", "employee", "hiring", "talent", "appraisal", "grievance",
           "hr", "induction", "attendance", "compensation", "engagement", "exit interviews", "statutory"],
    "Finance": ["audit", "budget", "variance", "reconciliation", "accounts payable", "accounts receivable", "gst",
                "tally", "forecast", "valuation", "financial", "dcf", "mis", "cash", "ind as"],
    "Healthcare": ["patient", "clinical", "hospital", "nursing", "pharmacy", "trial", "icu", "ward", "medication",
                   "vaccines", "ich-gcp", "emr", "infection", "radiology", "consent"],
    "Agriculture": ["crop", "soil", "seed", "irrigation", "farmer", "harvest", "pest", "fertiliser", "agronomy",
                    "yield", "drip", "horticulture", "organic", "field", "kvk"],
}
def keyword_counts(text):
    t = text.lower()
    return {dom: sum(len(re.findall(r"(?<![\w-])" + re.escape(k) + r"(?![\w-])", t)) for k in kws)
            for dom, kws in KEYWORDS.items()}
rows = {dom: keyword_counts(resume_text(rid)) for dom, rid in SAMPLE_IDS.items()}
counts = pd.DataFrame(rows).T
counts.index.name = "resume (true domain)"
display(counts)
print("Predicted by keyword count:", {dom: counts.loc[dom].idxmax() for dom in counts.index})

In [ ]:
import matplotlib.pyplot as plt
order = ["Engineering", "HR", "Finance", "Healthcare", "Agriculture"]
comp = df.groupby(["category", "source_type"]).size().unstack(fill_value=0).loc[order]
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.bar(order, comp["supplied_original"], color="#333333", label="supplied originals")
ax.bar(order, comp["synthetic"], bottom=comp["supplied_original"], color="#d9d9d9", edgecolor="#333333",
       hatch="//", label="synthetic (fictional)")
ax.set_ylabel("resumes"); ax.set_title("Resumes per domain (20 each, 100 in total)", fontsize=10)
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(comp.assign(total=comp.sum(axis=1)).to_string())